# Amazon ML Challenge 2026: Business Entity Resolution
**Team: PERCEPTRON**

This notebook automates the complete workflow on **Google Colab**:
1. Mount Google Drive or auto-download `student_resource.zip` directly via Unstop CDN.
2. Clone the private repository `amazon-ml-challenge-2026`.
3. Install dependencies.
4. Execute the multi-stage Entity Resolution pipeline (Blocking + Pairwise Features + LightGBM + F0.5 Calibration).
5. Run the official validator `utils/validate_submission.py`.
6. Package `PERCEPTRON_submission.zip` and copy it directly back to your Google Drive!

--- 
## Step 1: Mount Google Drive (Optional if downloading via CDN)
Mount Google Drive if you uploaded `amazon_ml_student_resource.zip` or `6ab10eb3b23ba_student_resource.zip` to your Drive.

In [ ]:
from google.colab import drive
import os, sys, shutil, zipfile
from pathlib import Path

try:
    drive.mount('/content/drive')
except Exception as e:
    print('Google Drive mount skipped or already mounted.')

--- 
## Step 2: Clone Private GitHub Repository
Enter your GitHub Personal Access Token (PAT) when prompted.
*(Create one in 10 seconds: GitHub -> Settings -> Developer Settings -> Personal access tokens -> Tokens (classic) -> Check 'repo' checkbox)*.

In [ ]:
import getpass

gh_token = getpass.getpass('Enter your GitHub Personal Access Token: ')

!rm -rf /content/amazon-ml-challenge-2026
!git clone https://{gh_token}@github.com/SabarishR08/amazon-ml-challenge-2026.git /content/amazon-ml-challenge-2026
%cd /content/amazon-ml-challenge-2026
!pip install -r code/business_entity_resolution/requirements.txt

--- 
## Step 3: Locate Dataset in Google Drive OR Download Directly via CDN URL
Checks Google Drive first. If not found in Drive, automatically downloads directly from the official Unstop CDN URL: `https://cdn.unstop.com/files/6ab10eb3b23ba_student_resource.zip`.

In [ ]:
zip_names = ['amazon_ml_student_resource.zip', '6ab10eb3b23ba_student_resource.zip']
zip_path = None

# 1. Check Google Drive
if os.path.exists('/content/drive/MyDrive'):
    for zn in zip_names:
        found = list(Path('/content/drive/MyDrive').rglob(zn))
        if found:
            zip_path = str(found[0])
            print(f'Found dataset zip in Google Drive: {zip_path}')
            break

# 2. If not found in Drive, download directly from official CDN
if not zip_path:
    cdn_url = 'https://cdn.unstop.com/files/6ab10eb3b23ba_student_resource.zip'
    zip_path = '/content/amazon_ml_student_resource.zip'
    print(f'Downloading directly from Unstop CDN: {cdn_url}...')
    !wget -q --show-progress -O /content/amazon_ml_student_resource.zip 'https://cdn.unstop.com/files/6ab10eb3b23ba_student_resource.zip'

print(f'Extracting {zip_path} directly onto fast NVMe disk (/content/amazon-ml-challenge-2026)...')
with zipfile.ZipFile(zip_path, 'r') as z:
    for item in z.infolist():
        if item.filename.startswith('__MACOSX') or item.filename.endswith('.DS_Store'):
            continue
        rel = item.filename
        if rel.startswith('student_resource/'):
            rel = rel[len('student_resource/'):]
        if not rel:
            continue
        target = os.path.join(os.getcwd(), rel)
        if item.is_dir():
            os.makedirs(target, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(item) as src, open(target, 'wb') as dst:
                dst.write(src.read())
print('Extraction complete! All train and test TSV files ready.')

--- 
## Step 4: Verify Extracted TSV Files

In [ ]:
!ls -lh dataset/train
!ls -lh dataset/test

--- 
## Step 5: Execute Complete Entity Resolution Pipeline
Runs candidate generation (blocking), pairwise feature calculation, and LightGBM classification with precision threshold calibration ($t = 0.68$).

In [ ]:
!python code/business_entity_resolution/src/pipeline.py \
    --train-dir dataset/train \
    --test-dir dataset/test \
    --output-dir output \
    --threshold 0.68

--- 
## Step 6: Validate and Package Final Submission
Validates `matching_results.tsv` and `candidate_pairs.tsv` against official competition rules and saves `PERCEPTRON_submission.zip` straight into your Google Drive.

In [ ]:
!python package_submission.py --team-name PERCEPTRON

# Copy to Google Drive if mounted
if os.path.exists('/content/drive/MyDrive'):
    drive_output_path = '/content/drive/MyDrive/PERCEPTRON_submission.zip'
    shutil.copy('PERCEPTRON_submission.zip', drive_output_path)
    print(f'Successfully copied final submission zip to: {drive_output_path}')